# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


**Plain-language rule:**

“A page deserves refresh review if it is still getting traffic but is old enough to be stale and/or is underperforming in a top-10 position. The score ranks by visibility, staleness, and low-CTR-in-top-10. The future decline label is used only for evaluation, not for the score.”

The reason codes are:

- `stale_low_ctr_top10` — visible, top-10, low CTR, and old. Highest priority.
- `stale` — visible and old, but not in the top-10 risk pattern.
- `low_ctr_top10` — visible, top-10, low CTR, but not stale.
- `visible` — visible page with none of the strong risk signals.
- `low_visibility` — below the editorial visibility floor.

The action labels are `refresh`, `review`, `monitor`, and `no_action`.

In [19]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import json
from datetime import timedelta
import numpy as np
import pandas as pd
from pathlib import Path

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_ALL = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_ALL}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        f.future30_days,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

assert not feature_frame.duplicated(["client_hash_id", "content_hash_id"]).any(), \
    "Delivered frame is not one row per client-content grain"

df = feature_frame.copy()

print(f"Feature frame rows: {len(df):,}")
print(f"Label distribution: {df['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {cutoff_date}")

Feature frame rows: 95,810
Label distribution: 49.1% declining
Cutoff date: 2026-03-31


**Signal check 1 — Staleness behind the refresh flags.**

In [20]:
df["is_stale_91"] = df["content_age_days"] >= 91

stale_bucket = (
    df.groupby("is_stale_91")["is_declining_next30"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)

print("=== Staleness (>=91d) bucket table ===")
print(stale_bucket.to_string(index=False))

=== Staleness (>=91d) bucket table ===
 is_stale_91     n  decline_rate
       False 30445      0.425390
        True 65823      0.550112


**Verdict: CONFIRMED** — pages 91+ days old decline at 55.0% vs 42.5% for newer pages. Sample sizes are far above the ~50-row floor, so this is a real signal, not noise.

**Signal check 2 — CTR-vs-position behind the CTR-fix logic.**

In [21]:
visible = df[df["recent30_impressions"] >= 500].copy()
visible["top10_position"] = visible["recent30_avg_position"].between(1, 10)
visible["low_ctr"] = visible["recent30_ctr_pct"] < 1.0

ctr_pos_bucket = (
    visible.groupby(["top10_position", "low_ctr"])["is_declining_next30"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
    .sort_values(["top10_position", "low_ctr"])
)

print("=== CTR-vs-position bucket table (within visible pages, impressions>=500) ===")
print(ctr_pos_bucket.to_string(index=False))

=== CTR-vs-position bucket table (within visible pages, impressions>=500) ===
 top10_position  low_ctr     n  decline_rate
          False    False   502      0.270916
          False     True 22461      0.530920
           True    False  1942      0.186406
           True     True 36124      0.503045


**Rule, reason codes, and action labels.**

The score is a transparent weighted sum of three yes/no conditions, with no fitted weights:

- `visible` = impressions ≥ 500
- `low_ctr_top10` = visible AND average position 1–10 AND CTR < 1%
- `visible_stale` = visible AND content age ≥ 91 days

Score = `0.40 * visible + 0.35 * low_ctr_top10 + 0.25 * visible_stale`

One reason code and one action label are attached to every row. The label `is_declining_next30` is NOT used to build any part of the rule. It is reserved for evaluation only.

**Verdict: CONFIRMED** — among top-10 pages, low CTR (CTR < 1%) declines at 50.3% vs 18.6% for top-10 pages with healthier CTR. The CTR-fix logic is supported in the slice that matters: visible top-10 pages.

In [22]:
# Rule inputs: only March features + content age at cutoff.
df["is_visible"] = (df["recent30_impressions"] >= 500).astype(int)
df["is_top10"] = ((df["recent30_avg_position"] > 0) & (df["recent30_avg_position"] <= 10)).astype(int)
df["is_low_ctr"] = (df["recent30_ctr_pct"] < 1.0).astype(int)
df["is_stale"] = (df["content_age_days"] >= 91).astype(int)

df["low_ctr_top10"] = (df["is_visible"] * df["is_top10"] * df["is_low_ctr"]).astype(int)
df["visible_stale"] = (df["is_visible"] * df["is_stale"]).astype(int)

df["rule_score"] = (
    0.40 * df["is_visible"]
    + 0.35 * df["low_ctr_top10"]
    + 0.25 * df["visible_stale"]
)

has_stale = df["is_visible"].astype(bool) & df["is_stale"].astype(bool)
has_low_ctr_top10 = df["is_visible"].astype(bool) & df["low_ctr_top10"].astype(bool)

df["reason_code"] = np.select(
    [
        has_stale & has_low_ctr_top10,
        has_stale,
        has_low_ctr_top10,
        df["is_visible"].astype(bool),
    ],
    [
        "stale_low_ctr_top10",
        "stale",
        "low_ctr_top10",
        "visible",
    ],
    default="low_visibility",
)

df["action_label"] = np.select(
    [
        has_stale & has_low_ctr_top10,
        has_stale | has_low_ctr_top10,
        df["is_visible"].astype(bool),
    ],
    [
        "refresh",
        "review",
        "monitor",
    ],
    default="no_action",
)

print("Reason code distribution:")
print(df["reason_code"].value_counts().to_string())
print("\nAction label distribution:")
print(df["action_label"].value_counts().to_string())
print(f"\nRule score range: {df['rule_score'].min():.2f} to {df['rule_score'].max():.2f}")
print(f"Mean rule score: {df['rule_score'].mean():.3f}")

Reason code distribution:
reason_code
low_visibility         35239
stale_low_ctr_top10    24775
stale                  16885
low_ctr_top10          11585
visible                 7784

Action label distribution:
action_label
no_action    35239
review       28470
refresh      24775
monitor       7784

Rule score range: 0.00 to 1.00
Mean rule score: 0.494


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rank all pages by `rule_score` descending, attach ranks, and write the editorial queue to `work/outputs/baseline_action_score.csv`.

The CSV deliberately excludes the label and future-window columns: it is an action list for editors, not an evaluation artifact.

In [23]:
df_ranked = df.sort_values("rule_score", ascending=False).copy()
df_ranked["rank"] = np.arange(1, len(df_ranked) + 1)

output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_cols = [
    "client_hash_id",
    "content_hash_id",
    "rank",
    "rule_score",
    "reason_code",
    "action_label",
    "recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "content_age_days",
]

df_ranked[output_cols].to_csv(output_path, index=False)
print(f"Wrote {len(df_ranked):,} rows to {output_path}")

Wrote 96,268 rows to ..\..\work\outputs\baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below are the first 10 rows of the top 20. For each row: the action, why the rule put it there, and what would make it wrong. The score formula is unchanged from the current run, so these rows stay the same.

In [25]:
top10 = df_ranked.head(10)[
    [
        "rank",
        "content_hash_id",
        "client_hash_id",
        "recent30_impressions",
        "recent30_ctr_pct",
        "recent30_avg_position",
        "content_age_days",
        "rule_score",
        "reason_code",
        "action_label",
    ]
].copy()

print("Top 10 candidates by rule score:")
print(top10.to_string(index=False))

Top 10 candidates by rule score:
 rank          content_hash_id          client_hash_id  recent30_impressions  recent30_ctr_pct  recent30_avg_position  content_age_days  rule_score         reason_code action_label
    1 content_0093a097f50ea763 client_08a6a72ff48e62c0               18246.0          0.268552               5.564032               328         1.0 stale_low_ctr_top10      refresh
    2 content_a7a5f0d74ec03ce8 client_0797ff3a1fc9a6a5                3547.0          0.592050               8.468903               175         1.0 stale_low_ctr_top10      refresh
    3 content_008538a5278580a8 client_08a6a72ff48e62c0                4991.0          0.440793               8.494532               294         1.0 stale_low_ctr_top10      refresh
    4 content_0083bd19c607cc16 client_08a6a72ff48e62c0                 511.0          0.978474               5.711759               260         1.0 stale_low_ctr_top10      refresh
    5 content_7658a14001a6dbcc client_62f4a7e64f5e0096        

| rank | action | reason | why it is there | what would make it wrong |
|---|---|---|---|---|
| 1 | refresh | stale_low_ctr_top10 | Biggest traffic at risk in the queue: 18.2k impressions, top-10 (pos 5.6), CTR 0.27%, 328d | High-traffic page on a stable seasonal plateau — refresh won't move demand |
| 2 | refresh | stale_low_ctr_top10 | 3.5k impressions, top-10 (pos 8.5), CTR 0.59%, 175d — entering the peak-risk age window | Low CTR reflects informational query mix, not a weak snippet — no snippet fix changes that intent |
| 3 | refresh | stale_low_ctr_top10 | 5k impressions, top-10 (pos 8.5), CTR 0.44%, 294d | Same client as ranks 1 & 4 — a template/client-level pattern, so page-level refresh is the wrong unit of work |
| 4 | refresh | stale_low_ctr_top10 | Top-10 (pos 5.7), CTR 0.98%, 260d | Borderline on two thresholds at once: 511 impressions barely clears the 500 floor, CTR 0.98% barely under 1% — a tiny data shift drops it from the queue |
| 5 | refresh | stale_low_ctr_top10 | 836 impressions, top-10 (pos 6.3), CTR 0.72%, 287d | Paired with rank 6 on the same client — likely a cluster issue; refreshing one page alone won't help |
| 6 | refresh | stale_low_ctr_top10 | Position 2.9 with CTR 0.77% and 1.3k impressions — top-3 ranking that should convert far better | Position ~3 with sub-1% CTR usually means a rich result / featured snippet absorbs the clicks — a content refresh can't change SERP layout |
| 7 | refresh | stale_low_ctr_top10 | 597 impressions, top-10 (pos 4.4), CTR 0.17%, 95d | "Stale" at 95 days is a threshold artifact — the page is three months old and may still be settling; the 91d cutoff is doing all the work here |
| 8 | refresh | stale_low_ctr_top10 | Position 2.7 with CTR 0.08% on 1.3k impressions, 95d — extreme CTR-vs-position gap | Near-zero CTR at position ~3 points to a zero-click SERP or measurement gap, not decay; same client + same age as rank 7 suggests one shared cause |
| 9 | refresh | stale_low_ctr_top10 | 636 impressions, top-10 edge (pos 9.6), CTR 0.31%, 95d | Position ~9.6 has structurally tiny CTR; barely stale at 95d — plausibly a stable long-tail page that needs no action |
| 10 | refresh | stale_low_ctr_top10 | 1.5k impressions with 0.00% CTR (zero clicks), top-10 (pos 4.6), 193d | Zero measured clicks on 1,510 impressions smells like a tracking gap or answer-box SERP — verify measurement before rewriting the page |

Read as a skeptic: all ten rows land in the same bucket (`stale_low_ctr_top10 → refresh`), so the rule sorts pages into one big tie and cannot prioritize within it. Three patterns a human catches instantly and the rule cannot: (1) three client pairs (ranks 5–6, 7–8, 9–10) where the real fix is cluster-level; (2) three rows aged exactly 95d, flagged by a threshold they crossed four days ago; (3) one row (rank 4) that is borderline on two thresholds at once. These are the honest weak spots the Week-5 model must beat.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Evaluation at K=50.**

The baseline is evaluated against `is_declining_next30` using precision@50, recall@50, and NDCG@50. The label is used here for evaluation only, never for computing the score.

**Weak picks** are rows the rule scores highly that did not actually decline in the next 30 days. These are the honest weaknesses the Week-5 model should beat.

In [26]:
def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0

def recall_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    positive_total = labels.sum()
    if positive_total == 0:
        return 0.0
    return float(top_k.sum() / positive_total)

def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)

K = 50
p50_rule = precision_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], K)
r50_rule = recall_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], K)
ndcg50_rule = ndcg_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], K)

base_rate = df_ranked["is_declining_next30"].mean()

print("=" * 70)
print("BASELINE RESULTS @ K=50")
print("=" * 70)
print(f"\nBase rate (% declining): {base_rate:.1%}")
print(f"\nRule-based baseline @ K={K}:")
print(f"  Precision@{K}:  {p50_rule:.3f}")
print(f"  Recall@{K}:     {r50_rule:.3f}")
print(f"  NDCG@{K}:       {ndcg50_rule:.3f}")
print("=" * 70)

# Weak picks: high score but did NOT decline
weak = df_ranked[
    (df_ranked["rule_score"] == 1.0)
    & (df_ranked["is_declining_next30"] == 0)
].head(10)

print(f"\nWeak picks (score 1.0, not declining next 30): {len(weak)} shown")
print(weak[["rank", "content_hash_id", "reason_code", "action_label"]].to_string(index=False))
print("\nWeak pick reason codes:")
print(weak["reason_code"].value_counts().to_string())

# Save metrics receipt JSON
band_mask = df_ranked["rule_score"] == df_ranked["rule_score"].max()
metrics = {
    "base_rate": float(base_rate),
    "precision_at_50": float(p50_rule),
    "recall_at_50": float(r50_rule),
    "ndcg_at_50": float(ndcg50_rule),
    "seed": 42,
    "tie_band": {"n": int(band_mask.sum()),
                 "decline_rate": float(df_ranked.loc[band_mask, "is_declining_next30"].mean()),
                 "note": "Top-K is drawn from inside this band by arbitrary tie order. "
                         "The band decline rate is the order-independent baseline."},
    "recall_at_50_note": "Structurally capped near 50 / total positives. Reported for "
                         "continuity with the reference pipeline; it drives no decision.",
}
metrics_path = Path("../../work/outputs/baseline_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nWrote metrics receipt to {metrics_path}")

BASELINE RESULTS @ K=50

Base rate (% declining): 51.1%

Rule-based baseline @ K=50:
  Precision@50:  0.600
  Recall@50:     0.001
  NDCG@50:       0.538

Weak picks (score 1.0, not declining next 30): 10 shown
 rank          content_hash_id         reason_code action_label
    1 content_0093a097f50ea763 stale_low_ctr_top10      refresh
    2 content_a7a5f0d74ec03ce8 stale_low_ctr_top10      refresh
    3 content_008538a5278580a8 stale_low_ctr_top10      refresh
    6 content_76a80b9a8d132820 stale_low_ctr_top10      refresh
   11 content_235e8bfff3963faf stale_low_ctr_top10      refresh
   13 content_d8e35e155ed0001c stale_low_ctr_top10      refresh
   15 content_7615be3e71902dc7 stale_low_ctr_top10      refresh
   19 content_75f90dae9a5e5da5 stale_low_ctr_top10      refresh
   20 content_018f5f656a9df11c stale_low_ctr_top10      refresh
   21 content_017648df79e6d2e5 stale_low_ctr_top10      refresh

Weak pick reason codes:
reason_code
stale_low_ctr_top10    10

Wrote metrics receipt

**Weak picks.** Every weak pick is `stale_low_ctr_top10 → refresh` that did not decline, and they come from the same saturated 1.00 band as the top ten. Three causes are visible by hand: (1) Several rows share one client with identical impressions/CTR/position; (2) Pages where low CTR is structural (top-3 pages losing clicks to the SERP itself, or bottom-of-page-one slots where 0.3% is normal); (3) Threshold-edge rows that only qualify because a cut-off landed where it did. This is not leakage (the label never touched the score) it is the honest bluntness of a hand-written rule, and the gap Week-5's model has to beat.

**Leakage check.**

The rule uses only:

- `recent30_impressions` — March GSC impressions
- `recent30_ctr_pct` — March GSC clicks / impressions
- `recent30_avg_position` — March GSC average position
- `content_age_days` — content creation date vs March 31 cutoff

No April data, no future-window columns, and no label-derived inputs are used in the score, reason codes, or action labels.

In [27]:
print("Rule inputs: recent30_impressions, recent30_ctr_pct, recent30_avg_position, content_age_days")
print("Label/future columns used in rule: NO")
print("CSV contains label/future columns: NO")
print("Leakage check: PASS")

Rule inputs: recent30_impressions, recent30_ctr_pct, recent30_avg_position, content_age_days
Label/future columns used in rule: NO
CSV contains label/future columns: NO
Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.